Convert & Preprocess

In [91]:
import librosa
import soundfile as sf
import numpy as np
from pydub import AudioSegment
from pydub.silence import split_on_silence

def convert_and_preprocess(input_path, output_dir):
    """Convert .m4a to .wav and apply basic preprocessing"""
    
    audio = AudioSegment.from_file(input_path, format="m4a")
    audio = audio.set_channels(1)
    audio = audio.set_frame_rate(16000)
    audio = audio.set_sample_width(2)
    
    wav_path = os.path.join(output_dir, "meeting_processed.wav")
    audio.export(wav_path, format="wav")
    
    return wav_path

Noise Reduction

In [92]:
import noisereduce as nr

def reduce_noise(wav_path, output_dir, noise_start=0.0, noise_end=0.5):
    y, sr = librosa.load(wav_path, sr=16000)
    
    noise_start_sample = int(noise_start * sr)
    noise_end_sample   = int(noise_end * sr)
    
    if noise_end_sample > len(y):
        raise ValueError("Noise sample window exceeds audio length. Adjust noise_start/noise_end.")
    
    noise_sample = y[noise_start_sample:noise_end_sample]
    
    rms = np.sqrt(np.mean(noise_sample**2))
    if rms > 0.02:
        print(f"Warning: Noise sample RMS is high ({rms:.4f}) — may contain speech, not silence.")
    
    y_denoised = nr.reduce_noise(y=y, sr=sr, y_noise=noise_sample, prop_decrease=0.75)
    
    out_path = os.path.join(output_dir, "denoised.wav")
    sf.write(out_path, y_denoised, sr)
    return out_path

VAD

In [93]:
import webrtcvad
import wave

def apply_vad(wav_path, output_dir, aggressiveness=2):
    """
    Optional — do NOT feed output into diarization (timestamps will misalign).
    aggressiveness: 0 (least) to 3 (most).
    """
    vad = webrtcvad.Vad(aggressiveness)
    
    with wave.open(wav_path, 'rb') as wf:
        sample_rate = wf.getframerate()
        pcm_data    = wf.readframes(wf.getnframes())
    
    frame_duration = 30  # ms
    frame_size     = int(sample_rate * frame_duration / 1000) * 2
    
    voiced_frames = []
    for i in range(0, len(pcm_data) - frame_size, frame_size):
        frame = pcm_data[i:i + frame_size]
        if len(frame) == frame_size and vad.is_speech(frame, sample_rate):
            voiced_frames.append(frame)
    
    out_path = os.path.join(output_dir, "vad_filtered.wav")
    with wave.open(out_path, 'wb') as out_wf:
        out_wf.setnchannels(1)
        out_wf.setsampwidth(2)
        out_wf.setframerate(sample_rate)
        out_wf.writeframes(b''.join(voiced_frames))
    
    return out_path

Diarization

In [94]:
from pyannote.audio import Pipeline
import torch
from huggingface_hub import login

def diarize_speakers(wav_path, hf_token, num_speakers=None):
    """
    Compatible with pyannote.audio 4.x
    DiarizeOutput.speaker_diarization is the actual Annotation object.
    """

    login(token=hf_token, add_to_git_credential=False)

    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1"
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pipeline = pipeline.to(device)
    print(f"        Running diarization on: {device}")

    params = {"num_speakers": num_speakers} if num_speakers else {}
    diarization = pipeline(wav_path, **params)

    # ✅ pyannote 4.x: extract Annotation from DiarizeOutput
    annotation = diarization.speaker_diarization

    segments = []
    for segment, _, speaker in annotation.itertracks(yield_label=True):
        segments.append({
            "speaker":  speaker,
            "start":    round(segment.start, 3),
            "end":      round(segment.end,   3),
            "duration": round(segment.end - segment.start, 3)
        })

    if not segments:
        raise ValueError("Diarization returned no segments — check audio quality or length.")

    return segments

Chunking

In [95]:
def chunk_audio_by_segments(wav_path, segments, output_dir, min_duration=1.5):
    """
    Slice audio into per-speaker utterance chunks.
    wav_path must be the SAME file passed to diarize_speakers()
    to keep timestamps aligned.
    """
    y, sr = librosa.load(wav_path, sr=16000)
    chunks_meta = []
    
    os.makedirs(output_dir, exist_ok=True)
    
    for i, seg in enumerate(segments):
        if seg["duration"] < min_duration:
            continue
        
        start_sample = int(seg["start"] * sr)
        end_sample   = int(seg["end"]   * sr)
        chunk        = y[start_sample:end_sample]
        
        filename   = f"chunk_{i:04d}_{seg['speaker']}_{seg['start']:.1f}-{seg['end']:.1f}.wav"
        chunk_path = os.path.join(output_dir, filename)
        sf.write(chunk_path, chunk, sr)
        
        chunks_meta.append({
            **seg,
            "chunk_file":    filename,
            "emotion_label": None,
            "intensity":     None,
            "valence":       None
        })
    
    return chunks_meta

Access Verification

In [96]:
from huggingface_hub import model_info, whoami, login
import os

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise EnvironmentError("HF_TOKEN not set. Run: os.environ['HF_TOKEN'] = 'hf_...'")

login(token=hf_token, add_to_git_credential=False)

user = whoami(token=hf_token)
print(f"Logged in as: {user['name']}")

# Only two repos needed for pyannote 4.x + diarization-3.1
repos = [
    "pyannote/speaker-diarization-3.1",
    "pyannote/segmentation-3.0",
]

all_ok = True
for repo in repos:
    try:
        model_info(repo, token=hf_token)
        print(f"✅ {repo}")
    except Exception as e:
        print(f"❌ {repo} — {e}")
        all_ok = False

if all_ok:
    print("\n✅ All repos accessible — safe to run pipeline")
else:
    print("\n❌ Accept conditions at each HF URL above, generate a fresh token, then retry")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in as: IsurindaDPerera27
✅ pyannote/speaker-diarization-3.1
✅ pyannote/segmentation-3.0

✅ All repos accessible — safe to run pipeline


In [97]:
# from huggingface_hub import login
# from pyannote.audio import Pipeline
# import torch, os

# login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

# pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")
# diarization = pipeline("./output/02_denoised/denoised.wav")

# print("Type:", type(diarization))
# print("\nAll attributes:")
# for attr in dir(diarization):
#     if not attr.startswith("__"):
#         print(f"  {attr}")

In [98]:
import os
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
hf_token    = os.environ.get("HF_TOKEN")
input_audio = "C:/Users/VICTUS/Desktop/AudioStream/1.m4a"
output_base = "./output"

if not hf_token:
    raise EnvironmentError("HF_TOKEN not set. Run: os.environ['HF_TOKEN'] = 'hf_...'")

# ── Directories ───────────────────────────────────────────────────────────────
convert_dir = os.path.join(output_base, "01_converted")
denoise_dir = os.path.join(output_base, "02_denoised")
vad_dir     = os.path.join(output_base, "03_vad")
chunks_dir  = os.path.join(output_base, "04_chunks")

for d in [convert_dir, denoise_dir, vad_dir, chunks_dir]:
    os.makedirs(d, exist_ok=True)

# ── Pipeline ──────────────────────────────────────────────────────────────────
try:
    print("Step 1: Converting M4A to WAV...")
    wav_file = convert_and_preprocess(input_audio, convert_dir)
    print(f"        Saved → {wav_file}")

    print("Step 2: Reducing noise...")
    denoised_file = reduce_noise(wav_file, denoise_dir)
    print(f"        Saved → {denoised_file}")

    print("Step 3: Diarizing speakers...")
    segments = diarize_speakers(denoised_file, hf_token)
    print(f"        Found {len(segments)} speaker segments")

    print("Step 4 (optional): Applying VAD for reference...")
    vad_file = apply_vad(denoised_file, vad_dir)
    print(f"        Saved → {vad_file}")

    print("Step 5: Chunking audio by speaker...")
    # Uses denoised_file — same as diarization input, keeps timestamps aligned
    chunks_metadata = chunk_audio_by_segments(denoised_file, segments, chunks_dir)
    print(f"        Created {len(chunks_metadata)} chunks")

    print("\n── First 5 chunks ───────────────────────────────────────")
    for chunk in chunks_metadata[:5]:
        print(f"  {chunk['chunk_file']}")
        print(f"    Speaker: {chunk['speaker']}  |  Duration: {chunk['duration']}s")

except FileNotFoundError as e:
    print(f"[ERROR] File not found: {e}")
except ValueError as e:
    print(f"[ERROR] Value error: {e}")
except Exception as e:
    print(f"[ERROR] Unexpected error: {e}")
    raise

Step 1: Converting M4A to WAV...
        Saved → ./output\01_converted\meeting_processed.wav
Step 2: Reducing noise...
        Saved → ./output\02_denoised\denoised.wav
Step 3: Diarizing speakers...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


        Running diarization on: cpu


c:\Users\VICTUS\miniconda3\envs\audioPipeline\lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1858.)
  std = sequences.std(dim=-1, correction=1)


        Found 403 speaker segments
Step 4 (optional): Applying VAD for reference...
        Saved → ./output\03_vad\vad_filtered.wav
Step 5: Chunking audio by speaker...
        Created 228 chunks

── First 5 chunks ───────────────────────────────────────
  chunk_0004_SPEAKER_01_15.0-18.4.wav
    Speaker: SPEAKER_01  |  Duration: 3.476s
  chunk_0006_SPEAKER_01_21.6-23.8.wav
    Speaker: SPEAKER_01  |  Duration: 2.211s
  chunk_0009_SPEAKER_01_26.3-30.9.wav
    Speaker: SPEAKER_01  |  Duration: 4.607s
  chunk_0010_SPEAKER_01_31.3-44.2.wav
    Speaker: SPEAKER_01  |  Duration: 12.876s
  chunk_0012_SPEAKER_01_45.9-52.0.wav
    Speaker: SPEAKER_01  |  Duration: 6.092s
